# Latency per passage (protocol §9)

Times every detector on the **same hardware (Colab T4)** and the **same unit of work: one retrieved group of 5 passages** for one question, which is what a defence sees at query time. 200 test n=1 groups (seed 42), 10 warm-up groups not timed, median reported.

- **DistilBERT** (augmented, seed 42): tokenise 5 (question, passage) pairs and score them.
- **Feature scorer:** two numbers. *Marginal*: the retriever already computed the passage and question embeddings, so only the three features and the logistic regression run. *Standalone*: also embeds the question and 5 passages with Contriever.
- **TrustRAG Stage-1:** k-means (k=2) plus pairwise cosine and ROUGE-L; marginal and standalone as above.

Writes `rag_trust/step3_features/latency.json`, which notebook 06 reads for Table 2. Timing only: nothing here changes a score or a threshold.

In [1]:
import os, sys, json, time, random, subprocess
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "rouge-score"], check=True)
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_ROOT = "/content/drive/MyDrive/rag_trust"
else:
    DRIVE_ROOT = os.environ["RAG_TRUST_ROOT"]
import numpy as np, torch, joblib
from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification
from sklearn.cluster import KMeans
from rouge_score import rouge_scorer
device = "cuda" if torch.cuda.is_available() else "cpu"
HW = torch.cuda.get_device_name(0) if device == "cuda" else "CPU"
print("device:", HW)
CONTRIEVER = os.environ.get("CONTRIEVER_NAME", "facebook/contriever")
DISTILBERT = os.environ.get("DISTILBERT_DIR", f"{DRIVE_ROOT}/step3_distilbert/model_aug_s42")
N_GROUPS, WARM, SEED = 200, 10, 42

Mounted at /content/drive
device: Tesla T4


In [2]:
rows = [json.loads(l) for l in open(f"{DRIVE_ROOT}/data/passages.jsonl")]
G = {}
for r in rows:
    if r["split"] == "test" and r["group_variant"] == "original" and r["n_poison"] == 1 and r["context_id"]:
        G.setdefault(r["context_id"], []).append(r)
G = {k: sorted(v, key=lambda r: r["rank"]) for k, v in G.items() if len(v) == 5}
keys = random.Random(SEED).sample(sorted(G), min(N_GROUPS + WARM, len(G)))
GROUPS = [G[k] for k in keys]
print(f"{len(GROUPS)} groups ({WARM} warm-up, {len(GROUPS) - WARM} timed)")

def sync():
    if device == "cuda": torch.cuda.synchronize()

def time_groups(fn):
    ms = []
    for i, grp in enumerate(GROUPS):
        sync(); t0 = time.perf_counter(); fn(grp); sync()
        if i >= WARM: ms.append((time.perf_counter() - t0) * 1000)
    return float(np.median(ms))

210 groups (10 warm-up, 200 timed)


## DistilBERT

In [3]:
dtok = AutoTokenizer.from_pretrained(DISTILBERT)
dmod = AutoModelForSequenceClassification.from_pretrained(DISTILBERT).to(device).eval()

@torch.no_grad()
def distilbert_group(grp):
    enc = dtok([r["question"] for r in grp], [r["passage"] for r in grp], truncation="only_second",
               max_length=256, padding=True, return_tensors="pt").to(device)
    with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(device == "cuda")):
        return torch.softmax(dmod(**enc).logits.float(), -1)[:, 1].cpu().numpy()

t_db = time_groups(distilbert_group)
print(f"DistilBERT: {t_db:.2f} ms per group of 5")

Loading weights:   0%|          | 0/104 [00:01<?, ?it/s]

DistilBERT: 9.31 ms per group of 5


## Contriever embeddings (needed by the feature scorer and TrustRAG)

In [4]:
ctok = AutoTokenizer.from_pretrained(CONTRIEVER)
cenc = AutoModel.from_pretrained(CONTRIEVER).to(device).eval()

@torch.no_grad()
def embed(texts):
    b = ctok(texts, padding=True, truncation=True, max_length=512, return_tensors="pt").to(device)
    h = cenc(**b).last_hidden_state
    m = b["attention_mask"].unsqueeze(-1).to(h.dtype)
    return ((h * m).sum(1) / m.sum(1)).float().cpu().numpy()      # masked mean pooling (as in the feature notebook)

def embed_group(grp):
    e = embed([grp[0]["question"]] + [r["passage"] for r in grp])
    return e[0], e[1:]

t_emb = time_groups(embed_group)
EMB = [embed_group(g) for g in GROUPS]                               # cached for the marginal timings
print(f"Contriever: {t_emb:.2f} ms per group (question + 5 passages)")

config.json:   0%|          | 0.00/619 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/321 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

...

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  438MB            

model.safetensors: downloading bytes:           |  0.00B            

Contriever: 50.09 ms per group (question + 5 passages)


## Feature scorer (features + frozen logistic regression)

In [5]:
fs = joblib.load(f"{DRIVE_ROOT}/data/frozen/feature_scorer.joblib")
scaler, lr = fs["scaler"], fs["lr"]
EMB_BY_ID = {id(g): e for g, e in zip(GROUPS, EMB)}

def features(q, P):
    f1 = P @ q                                                       # retrieval score (dot product)
    f2 = (f1 - f1.mean()) / (f1.std() + 1e-9)                        # z-score within the group
    X = P / np.linalg.norm(P, axis=1, keepdims=True)
    others = (X.sum(0, keepdims=True) - X) / 4
    f3 = 1 - (X * others).sum(1) / np.linalg.norm(others, axis=1)    # distance from the other four
    return np.c_[f1, f2, f3]

def featscorer_marginal(grp):
    q, P = EMB_BY_ID[id(grp)]
    return lr.predict_proba(scaler.transform(features(q, P)))[:, 1]

def featscorer_standalone(grp):
    q, P = embed_group(grp)
    return lr.predict_proba(scaler.transform(features(q, P)))[:, 1]

t_fs_m = time_groups(featscorer_marginal)
t_fs_s = time_groups(featscorer_standalone)
print(f"Feature scorer: marginal {t_fs_m:.3f} ms | standalone {t_fs_s:.2f} ms per group")

/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/ut

Feature scorer: marginal 0.570 ms | standalone 49.63 ms per group


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


## TrustRAG Stage-1 (simplified, as in `trustrag1-2.ipynb`)

In [6]:
COS_T, ROUGE_T = 0.85, 0.25
rs = rouge_scorer.RougeScorer(["rougeL"])
def mean_pair(m, fn): return float(np.mean([fn(a, b) for i, a in enumerate(m) for b in m[i + 1:]]))

def trustrag(P, texts):
    X = P / np.linalg.norm(P, axis=1, keepdims=True)
    lab = KMeans(2, n_init=5, random_state=42).fit_predict(X)
    qual = []
    for c in (0, 1):
        m = np.where(lab == c)[0]
        if len(m) < 2: continue
        cos = mean_pair(list(X[m]), lambda a, b: float(a @ b))
        if cos < COS_T: continue
        if mean_pair([texts[i] for i in m], lambda a, b: rs.score(a, b)["rougeL"].fmeasure) >= ROUGE_T:
            qual.append((cos, m))
    removed = np.zeros(len(P), bool)
    if qual: removed[max(qual, key=lambda x: x[0])[1]] = True
    return removed

def trustrag_marginal(grp): return trustrag(EMB_BY_ID[id(grp)][1], [r["passage"] for r in grp])
def trustrag_standalone(grp): return trustrag(embed_group(grp)[1], [r["passage"] for r in grp])

t_tr_m = time_groups(trustrag_marginal)
t_tr_s = time_groups(trustrag_standalone)
print(f"TrustRAG: marginal {t_tr_m:.2f} ms | standalone {t_tr_s:.2f} ms per group")

TrustRAG: marginal 5.17 ms | standalone 54.89 ms per group


## Save `latency.json`

In [7]:
LAT = {
    f"DistilBERT (aug s42), ms per passage, group of 5 ({HW})": round(t_db / 5, 3),
    f"Feature scorer, ms per passage, marginal given retriever embeddings ({HW})": round(t_fs_m / 5, 4),
    f"Feature scorer, ms per passage, standalone incl. Contriever ({HW})": round(t_fs_s / 5, 3),
    f"TrustRAG Stage-1, ms per passage, marginal given retriever embeddings ({HW})": round(t_tr_m / 5, 3),
    f"TrustRAG Stage-1, ms per passage, standalone incl. Contriever ({HW})": round(t_tr_s / 5, 3),
    "latency_method": f"median over {len(GROUPS) - WARM} test n=1 groups of 5 (seed {SEED}), {WARM} warm-up groups excluded",
}
os.makedirs(f"{DRIVE_ROOT}/step3_features", exist_ok=True)
json.dump(LAT, open(f"{DRIVE_ROOT}/step3_features/latency.json", "w"), indent=2)
print(json.dumps(LAT, indent=2))

{
  "DistilBERT (aug s42), ms per passage, group of 5 (Tesla T4)": 1.861,
  "Feature scorer, ms per passage, marginal given retriever embeddings (Tesla T4)": 0.1139,
  "Feature scorer, ms per passage, standalone incl. Contriever (Tesla T4)": 9.926,
  "TrustRAG Stage-1, ms per passage, marginal given retriever embeddings (Tesla T4)": 1.033,
  "TrustRAG Stage-1, ms per passage, standalone incl. Contriever (Tesla T4)": 10.977,
  "latency_method": "median over 200 test n=1 groups of 5 (seed 42), 10 warm-up groups excluded"
}
